In [0]:
from datetime import date
import time
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.functions import broadcast 

In [0]:
# ========================
# 0) Paramètres projet
# ========================
CATALOG = "workspace"
SCHEMA  = "projetdata"
VOLUME  = "projet"

VOLUME_ROOT  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PROJECT_ROOT = f"{VOLUME_ROOT}/projet"

RUN_DATE = date.today().isoformat()
RUN_ID   = RUN_DATE   # simple; acceptable pour idempotence

print("VOLUME_ROOT :", VOLUME_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DATE    :", RUN_DATE)

In [0]:
# ========================
# 1) Créer schema + volume (safe)
# ========================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print("Schema + Volume OK")

In [0]:
# ========================
# 2) Définir chemins
# ========================
RAW_ORDERS_DIR    = f"{PROJECT_ROOT}/raw/orders"
RAW_CUSTOMERS_DIR = f"{PROJECT_ROOT}/raw/customers"

BRONZE_MAIN   = f"{PROJECT_ROOT}/data/bronze/main/run_date={RUN_DATE}"
BRONZE_ENRICH = f"{PROJECT_ROOT}/data/bronze/enrich/run_date={RUN_DATE}"

SILVER_MAIN   = f"{PROJECT_ROOT}/data/silver/main_clean/run_date={RUN_DATE}"
SILVER_ENRICH = f"{PROJECT_ROOT}/data/silver/enrich_clean/run_date={RUN_DATE}"
SILVER_JOINED = f"{PROJECT_ROOT}/data/silver/joined/run_date={RUN_DATE}"

GOLD_MART    = f"{PROJECT_ROOT}/data/gold/marts/orders_enriched"
GOLD_AGG_DAY = f"{PROJECT_ROOT}/data/gold/aggregates/orders_kpis_daily"
GOLD_AGG_MON = f"{PROJECT_ROOT}/data/gold/aggregates/orders_kpis_monthly"
GOLD_EXPORT  = f"{PROJECT_ROOT}/data/gold/exports/orders_bi_parquet"

REPORT_DQ    = f"{PROJECT_ROOT}/reports/data_quality/run_date={RUN_DATE}"
REPORT_BENCH = f"{PROJECT_ROOT}/reports/benchmarks/run_date={RUN_DATE}"

for p in [
    RAW_ORDERS_DIR, RAW_CUSTOMERS_DIR,
    BRONZE_MAIN, BRONZE_ENRICH,
    SILVER_MAIN, SILVER_ENRICH, SILVER_JOINED,
    GOLD_MART, GOLD_AGG_DAY, GOLD_AGG_MON, GOLD_EXPORT,
    REPORT_DQ, REPORT_BENCH
]:
    dbutils.fs.mkdirs(p)

print("OK. RAW_ORDERS_DIR   :", RAW_ORDERS_DIR)
print("OK. RAW_CUSTOMERS_DIR:", RAW_CUSTOMERS_DIR)

In [0]:
# ========================
# 3) Vérifier les fichiers raw
# ========================
display(dbutils.fs.ls(RAW_ORDERS_DIR))
display(dbutils.fs.ls(RAW_CUSTOMERS_DIR))

In [0]:
# ========================
# 4) Lire Parquet raw
# ========================
orders_raw    = spark.read.parquet(RAW_ORDERS_DIR)        # lit les 12 fichiers
customers_raw = spark.read.parquet(RAW_CUSTOMERS_DIR)     # lit le fichier unique

print("orders_raw rows:", orders_raw.count())
print("customers_raw rows:", customers_raw.count())
orders_raw.printSchema()
customers_raw.printSchema()

display(orders_raw.limit(5))
display(customers_raw.limit(5))

In [0]:
# ========================
# 4b Profiling rapide 
# ========================
print("Distinct order_time:", orders_raw.select("order_time").distinct().count())
display(orders_raw.select("order_time").distinct().limit(20))

In [0]:
# ========================
# 5) Ingestion -> Bronze (Delta) + idempotence
# Idempotence: écriture dans run_date=YYYY-MM-DD + overwrite
# ========================
orders_bronze = (orders_raw
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("run_date", F.lit(RUN_DATE))
)

customers_bronze = (customers_raw
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("run_date", F.lit(RUN_DATE))
)

orders_bronze.write.format("delta").mode("overwrite").save(BRONZE_MAIN)
customers_bronze.write.format("delta").mode("overwrite").save(BRONZE_ENRICH)

print("Bronze written:")
print(" -", BRONZE_MAIN)
print(" -", BRONZE_ENRICH)

In [0]:
# ========================
# 6) Silver - nettoyage/standardisation + rejets (audit)
# - Schéma explicite (casts)
# - Nulls/doublons/dates
# - Standardisation chaînes
# - order_time: 00:00:00 -> NULL (non informatif)
# ========================
orders = spark.read.format("delta").load(BRONZE_MAIN)
customers = spark.read.format("delta").load(BRONZE_ENRICH)

# ---- Orders typed + standard
orders_typed = (orders
    .withColumn("order_id", F.col("order_id").cast("string"))
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("order_time", F.col("order_time").cast("string"))
    .withColumn("order_time",
        F.when(F.col("order_time") == F.lit("00:00:00"), F.lit(None)).otherwise(F.col("order_time"))
    )
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("discount", F.col("discount").cast("double"))
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))
    .withColumn("product_category", F.lower(F.trim(F.col("product_category"))))
    .withColumn("shipping_city", F.lower(F.trim(F.col("shipping_city"))))
    .withColumn("order_time_present", F.col("order_time").isNotNull().cast("int"))
)

valid_orders_cond = (
    F.col("order_id").isNotNull() &
    F.col("customer_id").isNotNull() &
    F.col("order_date").isNotNull() &
    (F.col("quantity") > 0) &
    (F.col("unit_price") >= 0) &
    (F.col("total_amount") >= 0) &
    (F.col("discount") >= 0)
)

orders_valid = (orders_typed
    .filter(valid_orders_cond)
    .dropDuplicates(["order_id"])
)

orders_rejects = (orders_typed
    .filter(~valid_orders_cond)
    .withColumn(
        "reject_reason",
        F.concat_ws("; ",
            F.when(F.col("order_id").isNull(), F.lit("order_id_null")),
            F.when(F.col("customer_id").isNull(), F.lit("customer_id_null")),
            F.when(F.col("order_date").isNull(), F.lit("order_date_null")),
            F.when(F.col("quantity").isNull(), F.lit("quantity_null")),
            F.when(F.col("quantity") <= 0, F.lit("quantity_non_positive")),
            F.when(F.col("unit_price") < 0, F.lit("unit_price_negative")),
            F.when(F.col("total_amount") < 0, F.lit("total_amount_negative")),
            F.when(F.col("discount") < 0, F.lit("discount_negative"))
        )
    )
)

# ---- Customers typed + standard
customers_typed = (customers
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("loyalty_points", F.col("loyalty_points").cast("int"))
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
    .withColumn("city", F.lower(F.trim(F.col("city"))))
    .withColumn("country", F.lower(F.trim(F.col("country"))))
    .withColumn("customer_type", F.lower(F.trim(F.col("customer_type"))))
    .withColumn("preferred_payment_method", F.lower(F.trim(F.col("preferred_payment_method"))))
)

valid_customers_cond = (
    F.col("customer_id").isNotNull() &
    (F.col("age").isNull() | ((F.col("age") >= 0) & (F.col("age") <= 120)))
)

customers_valid = (customers_typed
    .filter(valid_customers_cond)
    .dropDuplicates(["customer_id"])
)

customers_rejects = (customers_typed
    .filter(~valid_customers_cond)
    .withColumn(
        "reject_reason",
        F.concat_ws("; ",
            F.when(F.col("customer_id").isNull(), F.lit("customer_id_null")),
            F.when(F.col("age").isNotNull() & ((F.col("age") < 0) | (F.col("age") > 120)), F.lit("age_out_of_range"))
        )
    )
)

# Write Silver clean (idempotent par run_date car path inclut run_date)
orders_valid.write.format("delta").mode("overwrite").save(SILVER_MAIN)
customers_valid.write.format("delta").mode("overwrite").save(SILVER_ENRICH)

# Write rejects (audit)
REJECTS_DIR = f"{PROJECT_ROOT}/data/silver/rejects/run_date={RUN_DATE}"
ORDERS_REJ_DIR = f"{REJECTS_DIR}/orders"
CUSTOMERS_REJ_DIR = f"{REJECTS_DIR}/customers"
dbutils.fs.mkdirs(ORDERS_REJ_DIR)
dbutils.fs.mkdirs(CUSTOMERS_REJ_DIR)

orders_rejects.write.format("delta").mode("overwrite").save(ORDERS_REJ_DIR)
customers_rejects.write.format("delta").mode("overwrite").save(CUSTOMERS_REJ_DIR)

print("Silver written:")
print(" -", SILVER_MAIN)
print(" -", SILVER_ENRICH)
print("Rejects written:")
print(" -", ORDERS_REJ_DIR)
print(" -", CUSTOMERS_REJ_DIR)

In [0]:
# ========================
# 7) Joined - enrichissement multi-sources + features
# Optimisation: broadcast join si customers est petit
# Fix: éviter colonnes dupliquées run_date/run_id (Delta n'accepte pas)
# ========================
from pyspark.sql.functions import broadcast
from pyspark.sql import functions as F

orders_s = spark.read.format("delta").load(SILVER_MAIN)
customers_s = spark.read.format("delta").load(SILVER_ENRICH)

# 1) Renommer les colonnes techniques côté customers pour éviter doublons
customers_s2 = (customers_s
    .withColumnRenamed("run_id", "cust_run_id")
    .withColumnRenamed("run_date", "cust_run_date")
)

# 2) Broadcast si customers est petit
cust_count = customers_s2.count()
customers_for_join = broadcast(customers_s2) if cust_count < 5_000_000 else customers_s2

# 3) Join + features
joined = (orders_s
    .join(customers_for_join, on="customer_id", how="left")
    .withColumn("customer_tenure_days", F.datediff(F.col("order_date"), F.col("signup_date")))
    .withColumn("preferred_payment_match", (F.col("payment_method") == F.col("preferred_payment_method")).cast("int"))
    .withColumn("gross_amount", (F.col("quantity") * F.col("unit_price")).cast("double"))
    .withColumn(
        "age_bucket",
        F.when(F.col("age").isNull(), F.lit("unknown"))
         .when(F.col("age") < 25, F.lit("<25"))
         .when(F.col("age") < 35, F.lit("25-34"))
         .when(F.col("age") < 45, F.lit("35-44"))
         .when(F.col("age") < 60, F.lit("45-59"))
         .otherwise(F.lit("60+"))
    )
    .withColumn("customer_found", F.col("signup_date").isNotNull().cast("int"))
)

# 4) Optionnel: garder une trace run côté customers sans polluer (déjà renommée)
# Si tu ne veux pas garder, tu peux drop ces colonnes.
# joined = joined.drop("cust_run_id", "cust_run_date")

# 5) Write
joined.write.format("delta").mode("overwrite").save(SILVER_JOINED)
print("Joined written:", SILVER_JOINED)

In [0]:
# COMMAND ----------
# ========================
# 8) Gold - 3 outputs obligatoires
# - mart principale
# - agrégation temporelle (jour + mois)
# - export BI-ready (Parquet)
# Optimisation: partitionner par order_date (mart + agg day)
# ========================
df = spark.read.format("delta").load(SILVER_JOINED)

# (Option perf) réduire colonnes tôt
df_base = df.select(
    "order_id","customer_id","order_date","order_time","order_status","payment_method",
    "product_category","product_name","quantity","unit_price","discount","total_amount",
    "shipping_city",
    "city","country","customer_type","income_level","loyalty_points","age","gender",
    "signup_date","preferred_payment_method",
    "customer_tenure_days","preferred_payment_match","gross_amount","age_bucket","customer_found",
    "run_id","run_date"
)

# 8.1 Mart principale
mart = df_base.select(
    "order_id","customer_id","order_date","order_status","payment_method",
    "product_category","product_name","quantity","unit_price","discount","total_amount",
    "shipping_city","city","country","customer_type","income_level","loyalty_points",
    "age","gender","age_bucket",
    "customer_tenure_days","preferred_payment_match","gross_amount","customer_found",
    "run_id","run_date"
)

mart.write.format("delta").mode("overwrite").partitionBy("order_date").save(GOLD_MART)

# 8.2 Agrégation journalière
agg_daily = (df_base.groupBy("order_date")
    .agg(
        F.count("*").alias("orders"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.sum("total_amount").alias("revenue"),
        F.avg("total_amount").alias("avg_basket"),
        F.sum("discount").alias("discount_sum"),
        F.countDistinct("customer_id").alias("active_customers"),
        F.sum("customer_found").alias("orders_with_customer")
    )
)

agg_daily.write.format("delta").mode("overwrite").partitionBy("order_date").save(GOLD_AGG_DAY)

# 8.3 Agrégation mensuelle
df_month = df_base.withColumn("order_month", F.date_format(F.col("order_date"), "yyyy-MM"))
agg_monthly = (df_month.groupBy("order_month")
    .agg(
        F.count("*").alias("orders"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.sum("total_amount").alias("revenue"),
        F.avg("total_amount").alias("avg_basket"),
        F.sum("discount").alias("discount_sum"),
        F.countDistinct("customer_id").alias("active_customers")
    )
)

agg_monthly.write.format("delta").mode("overwrite").save(GOLD_AGG_MON)

# 8.4 Export BI-ready (Parquet)
mart.write.mode("overwrite").parquet(GOLD_EXPORT)

print("Gold written:")
print(" - MART    :", GOLD_MART)
print(" - AGG DAY :", GOLD_AGG_DAY)
print(" - AGG MON :", GOLD_AGG_MON)
print(" - EXPORT  :", GOLD_EXPORT)


In [0]:
# COMMAND ----------
# ========================
# 9) Data Quality (table résultats) — 8 checks
# Table requise: check_name, status, metric_value, threshold, run_id
# ========================
dfj = spark.read.format("delta").load(SILVER_JOINED)
total = dfj.count()

checks = []
def add_check(name, metric_value, threshold, passed):
    checks.append(Row(
        check_name=name,
        status="PASS" if passed else "FAIL",
        metric_value=float(metric_value),
        threshold=float(threshold),
        run_id=RUN_ID
    ))

# 1) order_id not null rate
null_order_id = dfj.filter(F.col("order_id").isNull()).count()
rate_order_id = 1 - (null_order_id/total if total else 1.0)
add_check("orders.order_id_not_null_rate", rate_order_id, 0.999, rate_order_id >= 0.999)

# 2) order_id uniqueness dup count
dup_order_id = dfj.groupBy("order_id").count().filter(F.col("count") > 1).count()
add_check("orders.order_id_unique_dup_count", dup_order_id, 0.0, dup_order_id == 0)

# 3) customer_id not null bad count
null_customer_id = dfj.filter(F.col("customer_id").isNull()).count()
add_check("orders.customer_id_not_null_bad_count", null_customer_id, 0.0, null_customer_id == 0)

# 4) total_amount >= 0 bad count
neg_total = dfj.filter(F.col("total_amount") < 0).count()
add_check("orders.total_amount_non_negative_bad_count", neg_total, 0.0, neg_total == 0)

# 5) quantity > 0 bad count
bad_qty = dfj.filter(F.col("quantity") <= 0).count()
add_check("orders.quantity_positive_bad_count", bad_qty, 0.0, bad_qty == 0)

# 6) unit_price >= 0 bad count
neg_price = dfj.filter(F.col("unit_price") < 0).count()
add_check("orders.unit_price_non_negative_bad_count", neg_price, 0.0, neg_price == 0)

# 7) tenure >= 0 (order_date >= signup_date) bad count
bad_tenure = dfj.filter(F.col("customer_tenure_days") < 0).count()
add_check("customers.tenure_non_negative_bad_count", bad_tenure, 0.0, bad_tenure == 0)

# 8) join rate (customers trouvés)
found_customer = dfj.filter(F.col("customer_found") == 1).count()
join_rate = (found_customer/total) if total else 0.0
add_check("join.customer_found_rate", join_rate, 0.95, join_rate >= 0.95)

# Bonus informatif (non bloquant): présence d'un order_time exploitable
time_present = dfj.filter(F.col("order_time").isNotNull()).count()
time_present_rate = (time_present/total) if total else 0.0
add_check("orders.order_time_present_rate", time_present_rate, 0.0, time_present_rate >= 0.0)

dq_df = spark.createDataFrame(checks)

dq_path = f"{REPORT_DQ}/dq_results_delta"
dq_df.write.format("delta").mode("overwrite").save(dq_path)

display(dq_df)
print("DQ saved:", dq_path)

# COMMAND ----------
# ========================
# 9bis) Data Quality report (Markdown) — interprétation
# ========================
md = []
md.append("# Data Quality Report\n")
md.append(f"- run_date: {RUN_DATE}\n- run_id: {RUN_ID}\n- total_joined_rows: {total}\n\n")
md.append("## Résultats des contrôles\n")

for r in dq_df.orderBy("check_name").collect():
    md.append(f"- {r['check_name']}: {r['status']} (metric={r['metric_value']}, threshold={r['threshold']})\n")

md.append("\n## Interprétation\n")
md.append("- Des valeurs négatives (prix/total) existent dans le raw; elles sont filtrées en Silver et tracées dans `data/silver/rejects/...`.\n")
md.append("- `order_time` n'est pas exploitable (NULL ou 00:00:00); on le neutralise en Silver et on ne produit pas de KPIs horaires.\n")

report_md_path = f"{REPORT_DQ}/data_quality_report.md"
dbutils.fs.put(report_md_path, "".join(md), overwrite=True)
print("DQ Markdown report saved:", report_md_path)


In [0]:
# COMMAND ----------
# ========================
# 10) Benchmarks & Optimisations (avant/après)
# - Durée (chrono)
# - Nb fichiers / taille outputs
# - Tentative de tuning shuffle partitions (si autorisée)
# ========================
import time

def try_set_conf(k, v):
    try:
        spark.conf.set(k, v)
        return True
    except Exception as e:
        print(f"WARNING: cannot set {k}={v} -> {e}")
        return False

def bench(label, df_to_action):
    t0 = time.time()
    df_to_action.count()
    secs = time.time() - t0
    return Row(run_id=RUN_ID, label=label, seconds=float(secs))

def list_all_files(path, max_depth=6):
    files = []
    stack = [(path, 0)]
    while stack:
        p, d = stack.pop()
        if d > max_depth:
            continue
        try:
            for x in dbutils.fs.ls(p):
                if x.path.endswith("/"):
                    stack.append((x.path, d+1))
                else:
                    files.append(x)
        except Exception:
            pass
    return files

def folder_metrics(path):
    files = list_all_files(path)
    return (len(files), int(sum(f.size for f in files)))

mart_gold = spark.read.format("delta").load(GOLD_MART)

rows = []
# A) baseline
rows.append(bench("A_default_filter_count", mart_gold.filter(F.col("order_date").isNotNull())))
rows.append(bench("A_default_groupby_category", mart_gold.groupBy("product_category").agg(F.sum("total_amount").alias("rev"))))

# B) tuned shuffle
ok = try_set_conf("spark.sql.shuffle.partitions", "16")  # recommandé 16/32
tag = "B_shuffle16" if ok else "B_shuffle16_not_allowed"

rows.append(bench(f"{tag}_filter_count", mart_gold.filter(F.col("order_date").isNotNull())))
rows.append(bench(f"{tag}_groupby_category", mart_gold.groupBy("product_category").agg(F.sum("total_amount").alias("rev"))))

bench_df = spark.createDataFrame(rows)

# métriques outputs
export_n, export_bytes = folder_metrics(GOLD_EXPORT)
mart_n, mart_bytes     = folder_metrics(GOLD_MART)
aggday_n, aggday_bytes = folder_metrics(GOLD_AGG_DAY)

metrics_df = spark.createDataFrame([
    Row(run_id=RUN_ID, metric="gold_export_n_files", value=float(export_n)),
    Row(run_id=RUN_ID, metric="gold_export_total_bytes", value=float(export_bytes)),
    Row(run_id=RUN_ID, metric="gold_mart_n_files", value=float(mart_n)),
    Row(run_id=RUN_ID, metric="gold_mart_total_bytes", value=float(mart_bytes)),
    Row(run_id=RUN_ID, metric="gold_agg_day_n_files", value=float(aggday_n)),
    Row(run_id=RUN_ID, metric="gold_agg_day_total_bytes", value=float(aggday_bytes)),
])

bench_path = f"{REPORT_BENCH}/benchmarks_delta"
bench_df.write.format("delta").mode("overwrite").save(f"{bench_path}/timings")
metrics_df.write.format("delta").mode("overwrite").save(f"{bench_path}/file_metrics")

display(bench_df)
display(metrics_df)
print("Benchmarks saved:", bench_path)
